# Desafio Pandas — Auditoria com Dependência entre Colunas

## Objetivo
Você recebeu um dataset de um marketplace. O problema é que várias colunas parecem corretas quando analisadas isoladamente, mas ficam incoerentes quando comparadas entre si.

Você deve limpar, padronizar, corrigir tipos, tratar nulos, remover duplicatas e criar colunas usando apenas os tópicos trabalhados em aula.

Use essas regras para encontrar inconsistências:

1. Se **status_pedido = pago**, então comprou deveria ser 1.
2. Se **status_pedido = cancelado**, então comprou deveria ser 0.
3. Se **comprou = 1**, então valor_carrinho deveria ser maior que 0 e qtd_itens maior que 0.
4. Se **qtd_itens = 0**, então valor_carrinho deveria ser 0.
5. Frete grátis só é esperado quando **valor_carrinho >= 200 ou uf = CE**.
6. **paginas_visitadas = 0** não faz sentido quando **tempo_site > 0**.
7. **tempo_site = 999** representa erro de captura.

## Setup do dataset
Execute a célula abaixo para criar o dataset bruto.

In [1]:
import pandas as pd

df = pd.read_csv('./data/marketplace.csv')
df.head()

,Unnamed: 0,User ID,Idade Cliente,Genero,UF,Categoria Produto,Tempo_Site,Paginas_Visitadas,Valor_Carrinho,Frete,Usou Cupom,Comprou,Status Pedido,Qtd Itens
0,0,1860.0,18,M,PE,mercado,0.0,4,300.0,gratis,sim,cancelado,NaN,10.0
1,1,NaN,trinta,Masculino,sp,mercado,999.0,5,NaN,25.5,não,nao,NaN,NaN
2,2,2044.0,18,Feminino,sp,Roup@,999.0,100,quatrocentos,gratis,NaN,cancelado,NaN,4.0
3,3,1121.0,25,NaN,sp,mercado,0.0,3,NaN,R$ 30,NAO,0,NaN,2.0
4,4,1466.0,30,M,CE,Roup@,2.0,3,150,gratis,sim,1,NaN,0.0


## Problema 1 — Diagnóstico inicial

Investigue o dataset bruto. Responda:

- Quantas linhas e colunas existem?
- Quais são os nomes das colunas?
- Quais tipos parecem incorretos?
- Onde existem valores nulos?
- O **describe()** ajuda em todas as colunas? Por quê?

In [3]:
print(f'Dataset marketplace possui {df.shape[0]} linhas e {df.shape[1]} colunas')
print(f'Lista de nomes das colunas: {df.columns}')

Dataset marketplace possui 4680 linhas e 14 colunas
Lista de nomes das colunas: Index(['Unnamed: 0', 'User ID', 'Idade Cliente', 'Genero', 'UF',
       'Categoria Produto', 'Tempo_Site', 'Paginas_Visitadas',
       'Valor_Carrinho', 'Frete', 'Usou Cupom', 'Comprou', 'Status Pedido',
       'Qtd Itens'],
      dtype='str')


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4680 entries, 0 to 4679
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         4680 non-null   int64  
 1   User ID            4610 non-null   float64
 2   Idade Cliente      4272 non-null   str    
 3   Genero             3665 non-null   str    
 4   UF                 3671 non-null   str    
 5   Categoria Produto  4274 non-null   str    
 6   Tempo_Site         4151 non-null   float64
 7   Paginas_Visitadas  4680 non-null   int64  
 8   Valor_Carrinho     4274 non-null   str    
 9   Frete              4023 non-null   str    
 10  Usou Cupom         3535 non-null   str    
 11  Comprou            4680 non-null   str    
 12  Status Pedido      3166 non-null   str    
 13  Qtd Itens          4039 non-null   float64
dtypes: float64(3), int64(2), str(9)
memory usage: 512.0 KB


Unnamed: 0 não precisaria existir, pois é uma repetição dos índices, User ID poderia ser int, Valor_Carrinho e Frete deveriam ser float, Usou Cupom, Comprou poderia ser booleano e Qtd Itens poderia ser int.

In [37]:
df.isnull().sum()

Unnamed: 0              0
User ID                70
Idade Cliente         408
Genero               1015
UF                   1009
Categoria Produto     406
Tempo_Site            529
Paginas_Visitadas       0
Valor_Carrinho        406
Frete                 657
Usou Cupom           1145
Comprou                 0
Status Pedido        1514
Qtd Itens             641
dtype: int64

In [17]:
df.describe()

,Unnamed: 0,User ID,Tempo_Site,Paginas_Visitadas,Qtd Itens
count,4680.00000,4610.000000,4151.000000,4680.000000,4039.000000
mean,2339.50000,1557.078308,132.769453,14.275000,3.300322
std,1351.14396,318.649910,329.410637,30.207281,3.191460
min,0.00000,1000.000000,-5.000000,0.000000,0.000000
25%,1169.75000,1278.000000,0.000000,2.000000,1.000000
50%,2339.50000,1566.000000,10.000000,4.000000,2.000000
75%,3509.25000,1837.000000,25.000000,6.000000,4.000000
max,4679.00000,2099.000000,999.000000,100.000000,10.000000


Não ajuda em todas as colunas. Primeiramente porque é inútil em colunas como Unnamed:0 e User ID e segundamente porque algumas colunas que deveriam ter valores numéricos estão como strings.

## Problema 2 — Padronizar nomes das colunas

Renomeie as colunas para um padrão mais profissional:

- letras minúsculas
- sem espaços
- nomes consistentes

Exemplo: **User ID** para **user_id**.

In [20]:
clean_df = df.copy()

clean_df.drop(columns='Unnamed: 0', inplace=True)
clean_df.head()

,User ID,Idade Cliente,Genero,UF,Categoria Produto,Tempo_Site,Paginas_Visitadas,Valor_Carrinho,Frete,Usou Cupom,Comprou,Status Pedido,Qtd Itens
0,1860.0,18,M,PE,mercado,0.0,4,300.0,gratis,sim,cancelado,NaN,10.0
1,NaN,trinta,Masculino,sp,mercado,999.0,5,NaN,25.5,não,nao,NaN,NaN
2,2044.0,18,Feminino,sp,Roup@,999.0,100,quatrocentos,gratis,NaN,cancelado,NaN,4.0
3,1121.0,25,NaN,sp,mercado,0.0,3,NaN,R$ 30,NAO,0,NaN,2.0
4,1466.0,30,M,CE,Roup@,2.0,3,150,gratis,sim,1,NaN,0.0


In [29]:
def clean_str(x):
    return x.strip().replace(' ', '_').lower() 

clean_str(' User ID ')

'user_id'

In [34]:
new_columns = {}
for col in clean_df.columns.values:
    new_columns[col] = clean_str(col)

new_columns

{'User ID': 'user_id',
 'Idade Cliente': 'idade_cliente',
 'Genero': 'genero',
 'UF': 'uf',
 'Categoria Produto': 'categoria_produto',
 'Tempo_Site': 'tempo_site',
 'Paginas_Visitadas': 'paginas_visitadas',
 'Valor_Carrinho': 'valor_carrinho',
 'Frete': 'frete',
 'Usou Cupom': 'usou_cupom',
 'Comprou': 'comprou',
 'Status Pedido': 'status_pedido',
 'Qtd Itens': 'qtd_itens'}

In [36]:
clean_df.rename(columns=new_columns, inplace=True)
clean_df.head()

,user_id,idade_cliente,genero,uf,categoria_produto,tempo_site,paginas_visitadas,valor_carrinho,frete,usou_cupom,comprou,status_pedido,qtd_itens
0,1860.0,18,M,PE,mercado,0.0,4,300.0,gratis,sim,cancelado,NaN,10.0
1,NaN,trinta,Masculino,sp,mercado,999.0,5,NaN,25.5,não,nao,NaN,NaN
2,2044.0,18,Feminino,sp,Roup@,999.0,100,quatrocentos,gratis,NaN,cancelado,NaN,4.0
3,1121.0,25,NaN,sp,mercado,0.0,3,NaN,R$ 30,NAO,0,NaN,2.0
4,1466.0,30,M,CE,Roup@,2.0,3,150,gratis,sim,1,NaN,0.0


## Problema 3 — Detectar strings vazias

Atenção: string vazia **""** não aparece como nulo no **isnull()**.

Investigue colunas como **genero**, **uf**, **categoria_produto**, **usou_cupom** e **status_pedido**.

In [ ]:
import numpy as np

def is_empty_str(x):
    return x == '' or x == ' ' or x == np.nan or x == None

clean_df.isna().sum()

user_id                70
idade_cliente         408
genero               1015
uf                   1009
categoria_produto     406
tempo_site            529
paginas_visitadas       0
valor_carrinho        406
frete                 657
usou_cupom           1145
comprou                 0
status_pedido        1514
qtd_itens             641
dtype: int64

In [47]:
mask = clean_df.map(lambda x: pd.isna(x) or (isinstance(x, str) and x.strip() == ''))
mask.sum()

user_id                70
idade_cliente         408
genero               1015
uf                   1009
categoria_produto     406
tempo_site            529
paginas_visitadas       0
valor_carrinho        406
frete                 657
usou_cupom           1145
comprou                 0
status_pedido        1514
qtd_itens             641
dtype: int64

In [48]:
clean_df.replace(r'^\s*$', np.nan, regex=True).isna().sum()

user_id                70
idade_cliente         408
genero               1015
uf                   1009
categoria_produto     406
tempo_site            529
paginas_visitadas       0
valor_carrinho        406
frete                 657
usou_cupom           1145
comprou                 0
status_pedido        1514
qtd_itens             641
dtype: int64

## Problema 4 — Padronizar colunas de texto

Padronize:

- **genero**
- **uf**
- **categoria_produto**
- **usou_cupom**
- **status_pedido**

Use **str.strip()**, **str.lower()**, **str.upper()** e **replace()**.

In [5]:
# sua solução aqui

## Problema 5 — Corrigir tipos numéricos

Corrija:

- **idade_cliente**
- **valor_carrinho**
- **frete**
- **qtd_itens**
- **comprou**

Cuidado com valores como **trinta**, **quatrocentos**, **R$ 400**, **R$400**, **gratis**, **sim**, **nao**, **cancelado**.

In [6]:
# sua solução aqui

## Problema 6 — Tratar valores inválidos de navegação

Trate:

- **tempo_site = -5**
- **tempo_site = 0**
- **tempo_site = 999**
- **paginas_visitadas = 0**
- **paginas_visitadas = 100**

Justifique suas escolhas.

In [7]:
# sua solução aqui

## Problema 7 — Dependência 1: compra e status do pedido

Encontre e corrija incoerências:

- **status_pedido = pago** mas **comprou = 0**
- **status_pedido = cancelado** mas **comprou = 1**

Explique qual coluna você escolheu como mais confiável.

In [8]:
# sua solução aqui

## Problema 8 — Dependência 2: compra, valor e quantidade

Encontre registros incoerentes:

- **comprou = 1** com **valor_carrinho = 0**
- **comprou = 1** com **qtd_itens = 0**
- **qtd_itens = 0** com **valor_carrinho > 0**

Corrija ou remova, justificando.

In [9]:
# sua solução aqui

## Problema 9 — Dependência 3: frete grátis suspeito

Pela regra de negócio, frete grátis só é esperado quando:

- **valor_carrinho >= 200**, ou
- **uf = CE**

Encontre registros suspeitos com **frete = 0**, **valor_carrinho < 200** e **uf != CE**.

In [10]:
# sua solução aqui

## Problema 10 — Remover duplicatas no momento certo

Remova duplicatas somente depois da padronização.

Compare a quantidade de duplicatas antes e depois da limpeza.

In [11]:
# sua solução aqui

## Problema 11 — Seleção, filtros e ordenação

Responda com filtros e ordenações:

1. Quais são os 10 maiores carrinhos?
2. Quais clientes compraram e usaram cupom?
3. Quais registros têm categoria **eletronicos** e carrinho acima de 200?
4. Mostre linhas e colunas específicas com **iloc()**.

In [12]:
# sua solução aqui

## Problema 12 — Criação de colunas

Crie:

- **ticket_por_item = valor_carrinho / qtd_itens**
- **valor_total = valor_carrinho + frete**
- **usuario_valioso = 1** quando **comprou = 1**, **valor_total >= 250** e **tempo_site >= 10**; caso contrário, **0**

Cuidado com divisão por zero.

In [13]:
# sua solução aqui

## Entrega final

Ao final, entregue:

1. Dataset limpo.
2. Quantidade inicial e final de linhas.
3. Lista de problemas encontrados.
4. Decisões tomadas.
5. Três insights simples obtidos por filtros/ordenação.